# Full manual lineage: Mortgage complaint response

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "01_consumer_complaints"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
QUERY = [
    ("date_received_min", "2024-01-01"),
    ("date_received_max", "2026-01-01"),
    ("product", "Mortgage"),
    ("state", "PA"), ("state", "VA"), ("state", "MD"),
    ("state", "DC"), ("state", "NJ"),
    ("format", "csv"),
]
FIELDS = ["Date received", "Product", "Issue", "Company", "State",
          "Submitted via", "Date sent to company",
          "Company response to consumer", "Timely response?", "Complaint ID"]
USE_PINNED_RAW = True  # Set False only when deliberately acquiring a fresh publisher response.
PINNED_SHA = "37feb23d5c38a641731a592c36dfebb1664097ae03c21bcd03bc39841ffc10d6"
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RESPONSES = RAW_DIR / "publisher_responses"
RESPONSES.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Selection:", QUERY)
print("Pinned publisher SHA-256:", PINNED_SHA)

def acquire_response():
    if USE_PINNED_RAW:
        digest = PINNED_SHA
        body_path = RESPONSES / (digest + ".csv")
        metadata_path = RESPONSES / (digest + ".json")
        if not body_path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError("Pinned publisher response or metadata is missing")
        body = body_path.read_bytes()
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if hashlib.sha256(body).hexdigest() != digest or metadata["sha256"] != digest:
            raise ValueError("Pinned CFPB response checksum mismatch")
        if metadata["query"] != [list(pair) for pair in QUERY]:
            raise ValueError("Pinned CFPB selection does not match this notebook")
        return body, metadata
    response = requests.get(SOURCE_URL, params=QUERY, timeout=(20, 90))
    response.raise_for_status()
    body = response.content
    if not body.startswith(b"Date received,"):
        raise ValueError("CFPB did not return the expected CSV")
    digest = hashlib.sha256(body).hexdigest()
    metadata = {"publisher": "Consumer Financial Protection Bureau",
                "direct_url": response.url, "query": QUERY,
                "accessed_at": datetime.now(UTC).isoformat(),
                "bytes": len(body), "sha256": digest,
                "content_type": response.headers.get("Content-Type"),
                "publisher_last_modified": response.headers.get("Last-Modified")}
    (RESPONSES / (digest + ".csv")).write_bytes(body)
    (RESPONSES / (digest + ".json")).write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return body, metadata


Project: projects/consumer_complaints
Publisher: https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/
Selection: [('date_received_min', '2024-01-01'), ('date_received_max', '2026-01-01'), ('product', 'Mortgage'), ('state', 'PA'), ('state', 'VA'), ('state', 'MD'), ('state', 'DC'), ('state', 'NJ'), ('format', 'csv')]
Pinned publisher SHA-256: 37feb23d5c38a641731a592c36dfebb1664097ae03c21bcd03bc39841ffc10d6


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview. The CFPB CSV export may include rows dated on the requested maximum date; the visible import cell excludes any dates outside the stated 2024-2025 window and reports how many were excluded. It normalizes the API timestamps to UTC calendar dates, matching the project's original day-level fields.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
response_body, download_report = acquire_response()
raw = pd.read_csv(BytesIO(response_body), usecols=FIELDS, dtype="string")
if raw.empty or raw["Complaint ID"].isna().any() or raw["Complaint ID"].duplicated().any():
    raise ValueError("CFPB source IDs must be present and unique")
if not raw["Product"].eq("Mortgage").all() or not raw["State"].isin(["PA", "VA", "MD", "DC", "NJ"]).all():
    raise ValueError("CFPB API response includes records outside the requested cohort")
received = pd.to_datetime(raw["Date received"], utc=True, errors="raise")
in_window = received.ge("2024-01-01") & received.lt("2026-01-01")
outside_window = int((~in_window).sum())
raw = raw.loc[in_window].copy()
if raw.empty:
    raise ValueError("No CFPB records in the stated 2024-2025 cohort")
raw["Date received"] = pd.to_datetime(raw["Date received"], utc=True, errors="raise").dt.strftime("%Y-%m-%d")
raw["Date sent to company"] = pd.to_datetime(raw["Date sent to company"], utc=True, errors="coerce").dt.strftime("%Y-%m-%d")
print("Publisher rows outside the stated date window, excluded:", outside_window)
SNAPSHOT_DATE = str(download_report["accessed_at"])[:10]
print("Source date:", SNAPSHOT_DATE)
print("Publisher response SHA-256:", download_report["sha256"])
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Publisher rows outside the stated date window, excluded: 5
Source date: 2026-09-26
Publisher response SHA-256: 37feb23d5c38a641731a592c36dfebb1664097ae03c21bcd03bc39841ffc10d6
Imported rows: 6558 columns: 10


,Date received,Product,Issue,Company,State,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-01-02,Mortgage,Struggling to pay mortgage,WELLS FARGO & COMPANY,NJ,Web,2024-01-02,Closed with explanation,Yes,8091735
1,2024-01-02,Mortgage,Trouble during payment process,TRUIST FINANCIAL CORPORATION,MD,Referral,2024-01-02,Closed with explanation,Yes,8089203
2,2024-01-02,Mortgage,Trouble during payment process,WELLS FARGO & COMPANY,MD,Web,2024-01-02,Closed with explanation,Yes,8090882


## 3 Transform and clean

The original cleaning function is embedded below. Run it, inspect the fields and missing values, and confirm the record grain before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def consumer_complaints(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(
        x, ["complaint_id", "date_received", "product", "company", "state", "timely_response"]
    )
    x = x.drop_duplicates()
    if x["complaint_id"].isna().any():
        raise ValueError("Complaint IDs must be observed")
    if x["complaint_id"].duplicated().any():
        raise ValueError("Conflicting complaint IDs require source reconciliation")
    x["date_received"] = pd.to_datetime(x["date_received"], errors="coerce")
    if "date_sent_to_company" in x:
        x["date_sent"] = pd.to_datetime(x["date_sent_to_company"], errors="coerce")
    else:
        x["date_sent"] = pd.NaT
    x["days_to_send"] = (x["date_sent"] - x["date_received"]).dt.days
    x["timely_flag"] = (
        x["timely_response"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": 1, "no": 0})
        .astype("Int64")
    )
    disputed = x.get("consumer_disputed", pd.Series(pd.NA, index=x.index, dtype="string"))
    x["disputed_flag"] = (
        disputed.astype("string").str.lower().map({"yes": 1, "no": 0}).astype("Int64")
    )
    x["complaint_month"] = x["date_received"].dt.to_period("M").dt.to_timestamp()
    x["company_volume"] = x.groupby("company")["complaint_id"].transform("size")
    x["company_timely_rate"] = x.groupby("company")["timely_flag"].transform("mean").round(4)
    x["product_timely_rate"] = x.groupby("product")["timely_flag"].transform("mean").round(4)
    # Historical dispute reporting is absent from recent data; never impute No.
    # This two-component review scenario is not a validated complaint-risk model.
    x["response_risk_score"] = (
        50 * (1 - x["timely_flag"]) + 50 * x["days_to_send"].clip(0, 10).div(10)
    ).round(1)
    x["review_band"] = pd.cut(
        x["response_risk_score"],
        [-np.inf, 25, 50, np.inf],
        labels=["Monitor", "Review", "Escalate"],
    )
    keep = [
        "complaint_id",
        "date_received",
        "date_sent",
        "complaint_month",
        "product",
        "issue",
        "company",
        "state",
        "submitted_via",
        "company_response_to_consumer",
        "days_to_send",
        "timely_flag",
        "disputed_flag",
        "company_volume",
        "company_timely_rate",
        "product_timely_rate",
        "response_risk_score",
        "review_band",
    ]
    return x.reindex(columns=keep).drop_duplicates("complaint_id")

clean = consumer_complaints(raw)
print("Clean rows:", len(clean), "Columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 6558 Columns: 18


,complaint_id,date_received,date_sent,complaint_month,product,issue,company,state,submitted_via,company_response_to_consumer,days_to_send,timely_flag,disputed_flag,company_volume,company_timely_rate,product_timely_rate,response_risk_score,review_band
0,8091735,2024-01-02,2024-01-02,2024-01-01,Mortgage,Struggling to pay mortgage,WELLS FARGO & COMPANY,NJ,Web,Closed with explanation,0,1,<NA>,545,1.0,0.988,0.0,Monitor
1,8089203,2024-01-02,2024-01-02,2024-01-01,Mortgage,Trouble during payment process,TRUIST FINANCIAL CORPORATION,MD,Referral,Closed with explanation,0,1,<NA>,174,1.0,0.988,0.0,Monitor
2,8090882,2024-01-02,2024-01-02,2024-01-01,Mortgage,Trouble during payment process,WELLS FARGO & COMPANY,MD,Web,Closed with explanation,0,1,<NA>,545,1.0,0.988,0.0,Monitor


## 4 Calculate

The original analytical functions are embedded below. Run the cell and inspect the response-queue output.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def monthly_queue(frame: pd.DataFrame, minimum_responses: int = 20, target_timely_rate: float = 0.95) -> pd.DataFrame:
    """Company-product calendar-month Wilson review, matching the supplied SQL."""
    grouped = (frame.groupby(["company", "product", "complaint_month"], dropna=False)
               .agg(complaints=("complaint_id", "size"),
                    known_responses=("timely_flag", "count"),
                    timely_responses=("timely_flag", "sum"))
               .reset_index().rename(columns={"complaint_month": "month"}))
    n = grouped["known_responses"].astype(float)
    p = grouped["timely_responses"].astype(float).div(n.where(n.ne(0)))
    z = 1.96
    denominator = 1 + z * z / n.where(n.ne(0))
    center = (p + z * z / (2 * n.where(n.ne(0)))) / denominator
    margin = z * np.sqrt(p * (1 - p) / n.where(n.ne(0)) + z * z / (4 * n.where(n.ne(0)) ** 2)) / denominator
    grouped["timely_rate"] = p
    grouped["wilson_low"] = (center - margin).clip(lower=0)
    grouped["wilson_high"] = (center + margin).clip(upper=1)
    grouped["response_coverage"] = n.div(grouped["complaints"])
    grouped["below_target_signal"] = ((n >= minimum_responses) & (grouped["wilson_high"] < target_timely_rate)).astype(int)
    grouped = grouped.sort_values(["company", "product", "month"]).reset_index(drop=True)
    prior_month = grouped.groupby(["company", "product"], dropna=False)["month"].shift()
    prior_signal = grouped.groupby(["company", "product"], dropna=False)["below_target_signal"].shift().fillna(0)
    grouped["persistent_signal"] = ((grouped["below_target_signal"] == 1) & (prior_signal == 1) & (grouped["month"] == prior_month + pd.offsets.MonthBegin(1))).astype(int)
    grouped["target_timely_rate"] = target_timely_rate
    return grouped

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["response_queue_intervals"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 1, 'bootstrap_gap': 1, 'feature_diagnostics': 4, 'temporal_diagnostics': 24, 'threshold_sensitivity': 5, 'decision_queue': 20, 'response_queue_intervals': 1671}


,company,product,month,complaints,known_responses,timely_responses,timely_rate,wilson_low,wilson_high,response_coverage,below_target_signal,persistent_signal,target_timely_rate
0,1st Fidelity Loan Servicing,Mortgage,2025-07-01,1,1,0,0.0,0.000000,0.793457,1.0,0,0,0.95
1,21ST MORTGAGE CORP.,Mortgage,2024-02-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
2,21ST MORTGAGE CORP.,Mortgage,2024-07-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
3,21ST MORTGAGE CORP.,Mortgage,2025-07-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
4,21ST MORTGAGE CORP.,Mortgage,2025-08-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
report = profile(clean, key="complaint_id")
assert report["passed"], report["notes"]
assert clean["complaint_id"].notna().all() and clean["complaint_id"].is_unique
assert clean["timely_flag"].dropna().isin([0, 1]).all()
print("Current source rows:", len(raw), "Clean rows:", len(clean))
print("Quality:", report)


Current source rows: 6558 Clean rows: 6558
Quality: {'rows': 6558, 'columns': 18, 'duplicate_rows': 0, 'missing_cells': 6558, 'missing_rate': 0.055556, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after Check passes. It saves the checked extract, quality report, analysis tables, and source manifest. Save never writes into an existing run folder. If the dated folder already exists, it creates a new timestamp-and-random-suffix run folder, even when the source fingerprint is unchanged. This keeps the original result available for comparison. Use the exact output folder printed by Save for subsequent SQL and Tableau checks; a partial run must be investigated, not merged into a later run.

**Check before continuing:** confirm the printed folder contains the current source digest, row count, and every named output. Compare with the untouched pinned result before claiming a match.

In [ ]:
# Preserve every existing saved result, even when the source bytes are identical.
from uuid import uuid4
assert report["passed"], "Run Check successfully before Save"
assert report["rows"] == len(clean), "The checked row count changed; rerun Check"
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    run_stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{SNAPSHOT_DATE}-run-{run_stamp}-{uuid4().hex[:8]}")
# Exclusive creation also stops an unexpected name collision; never merge runs.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "Consumer Financial Protection Bureau",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": download_report["sha256"]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps(download_report, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `create_table.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `load_extract.sql`, then run `check_load.sql` and `review_queries.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [ ]:
display(analysis_outputs["response_queue_intervals"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,company,product,month,complaints,known_responses,timely_responses,timely_rate,wilson_low,wilson_high,response_coverage,below_target_signal,persistent_signal,target_timely_rate
0,1st Fidelity Loan Servicing,Mortgage,2025-07-01,1,1,0,0.0,0.000000,0.793457,1.0,0,0,0.95
1,21ST MORTGAGE CORP.,Mortgage,2024-02-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
2,21ST MORTGAGE CORP.,Mortgage,2024-07-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
3,21ST MORTGAGE CORP.,Mortgage,2025-07-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
4,21ST MORTGAGE CORP.,Mortgage,2025-08-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
5,21ST MORTGAGE CORP.,Mortgage,2025-10-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
6,A&D Mortgage LLC,Mortgage,2025-12-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
7,A1 Holding Group LLC,Mortgage,2025-11-01,1,1,0,0.0,0.000000,0.793457,1.0,0,0,0.95
8,A1 Lending Team LLC,Mortgage,2024-12-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95
9,"ADVISORS MORTGAGE GROUP, LLC",Mortgage,2024-08-01,1,1,1,1.0,0.206543,1.000000,1.0,0,0,0.95


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [9]:
DECISION_TEXT = "For the 2026-09-26 CFPB snapshot, use analysis/response_queue_intervals.csv to prioritize manual review of any flagged company-month groups. The quality_report.json check passed. A flag is a screening signal, not proof of poor service."
INTERPRETATION_LIMIT = "Published complaints are not representative of all consumers, and complaint counts do not measure market share or prevalence."
EVIDENCE_FILES = ["quality_report.json", "analysis/response_queue_intervals.csv"]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)


Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/response_queue_intervals.csv']
Decision: For the 2026-09-26 CFPB snapshot, use analysis/response_queue_intervals.csv to prioritize manual review of any flagged company-month groups. The quality_report.json check passed. A flag is a screening signal, not proof of poor service.
Limit: Published complaints are not representative of all consumers, and complaint counts do not measure market share or prevalence.
